# NB06 — Permutation Importance

Audits the trained Random Forest from NB04. The impurity-based feature
importance reported in NB04 §5 is structurally biased toward continuous,
high-cardinality features, which makes the Tech vs Sector vs LLM block
comparison unreliable. This notebook recomputes feature importance via
permutation on the held-out test sets, so the comparison is grounded in
actual OOS predictive contribution rather than tree-construction
artifacts.

**Sections**
1. Load model, rebuild feature matrix, sanity check
2. Pooled OOS permutation importance (test_2024 + test_2025)
3. Block-level summary: permutation vs impurity-based
4. Per-year decomposition (regime stability)
5. Outputs for the paper appendix

The code is intentionally short — the reusable bits (`run_permutation`,
`block_summary`, `build_per_feature_table`) live in
`src/llm_agent/importance.py`; this notebook is just the experimental
driver.

## 1. Load model and rebuild the feature matrix

The pickled bundle contains the fitted Pipeline plus the exact feature
order it was trained on. We rebuild the matrix using
`features.build_feature_matrix(arm='C')` so this notebook depends on no
intermediate artifacts beyond what NB00 produces.

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path('..').resolve()))
import config
from src.llm_agent.data_loader import load_sessions, load_prices, load_llm_signals
from src.llm_agent.features    import build_feature_matrix
from src.llm_agent.importance  import (
    run_permutation, block_summary, build_per_feature_table,
)

bundle = joblib.load(config.MODELS_DIR / 'event_study_best.pkl')
pipe, feats = bundle['model'], bundle['features']
print(f'model: {bundle["name"]}, target: {bundle["main_window"]}, n_features: {len(feats)}')
print(f'CV AUC (held-out, pre-refit): {bundle["cv_auc_tuned"]:.4f}')
print(f'val AUC (held-out, pre-refit): {bundle["val_auc_tuned"]:.4f}')

In [ ]:
sessions = load_sessions()
prices   = load_prices()
sigs     = load_llm_signals()

base = build_feature_matrix(sessions, prices, sigs, arm='C')

# Attach binary target = sign of car_2_11 (NB04's target)
car = pd.read_parquet(config.CLEAN_DIR / 'car_labels.parquet')
base = base.merge(car[['session_id', 'car_2_11', 'sigma_e']], on='session_id', how='left')
base['y'] = (base['car_2_11'] > 0).astype('Int64')

print(f'feature matrix: {base.shape}')
print(f'NaN-free rows:  {base.dropna(subset=feats + ["y"]).shape[0]}')

**Sanity check.** The pickled model was refit on train ∪ val (NB04 §5).
That means our 2023 predictions are *in-sample* and should give AUC ≈
0.84 — substantially higher than the held-out `val_auc_tuned = 0.601`
reported above. A diff of ~0.24 confirms the feature matrix is correctly
aligned to the trained model.

In [ ]:
val = base[base.fiscal_year == 2023].dropna(subset=feats + ['car_2_11', 'sigma_e'])
val = val[val['car_2_11'].abs() >= 0.5 * val['sigma_e'] * np.sqrt(10)]   # tail filter
auc_in = roc_auc_score(
    val['y'].astype(int),
    pipe.predict_proba(val[feats].astype(float))[:, 1],
)
print(f'in-sample val 2023 AUC: {auc_in:.4f}   (expected ≈ 0.84)')

## 2. Pooled OOS permutation importance

Pool 2024 and 2025 test events (n ≈ 360) and shuffle each feature
independently 30 times, scoring with AUC. Δ AUC > 0 ⇒ feature is being
used; Δ AUC ≤ 0 ⇒ feature is noise or redundant on this data.

In [ ]:
test_pool = base[base.fiscal_year.isin([2024, 2025])]
print(f'pooled test n: {len(test_pool)}, class balance: {test_pool["y"].mean():.3f}')

perm_pool = run_permutation(pipe, feats, test_pool, n_repeats=30, random_state=0)
perm_pool.round(4)

**Reading the table.** `log_dollar_vol` and `evasion` are the only two
features whose Δ AUC clearly separates from zero. `evasion` ranks
second overall — ahead of all four other technical features — and is
the only LLM feature with a clear positive contribution. The four
remaining LLM features (sentiment, confidence, certainty, tone,
guidance) and three of the five technical features (rsi_7, mom_5d,
macd_hist) carry weakly negative permutation importance, which means
they contribute essentially nothing to OOS ranking and may be
redundant given `log_dollar_vol` and `evasion`.

## 3. Block summary — permutation vs impurity

Aggregate the per-feature numbers up to the three blocks (Tech /
Sector / LLM) used in the paper, and compare against the impurity
shares from NB04.

In [ ]:
blk = block_summary(perm_pool, feats, pipe=pipe)
blk.round(2)

**Headline.** The LLM block carries **36.6%** of pooled OOS permutation
importance versus **11.7%** under impurity-based scoring — a 3.1×
difference. The Tech block falls from 84.8% (impurity) to 47.0%
(permutation); Sector rises from 3.5% to 16.4%. The reason is mechanical:
impurity favors features with many candidate split points, which
inflates `log_dollar_vol`, `vol_10d`, and `mom_5d` and deflates the
binary sector dummies and the bounded-integer LLM features. The
permutation ranking is the more honest comparison.

## 4. Per-year decomposition

The headline ΔSharpe in NB05 is +0.13 in 2024 and +1.07 in 2025 — much
larger in 2025. If the LLM block were the main source of that lift,
its permutation importance should also be larger in 2025. Run the
decomposition and check.

In [ ]:
per_year = {}
for year in [2024, 2025]:
    df_y = base[base.fiscal_year == year]
    perm_y = run_permutation(pipe, feats, df_y, n_repeats=30, random_state=0)
    per_year[year] = block_summary(perm_y, feats, pipe=pipe)[['block','perm_delta_metric']]

pd.concat(
    [df.set_index('block').rename(columns={'perm_delta_metric': str(yr)})
     for yr, df in per_year.items()],
    axis=1,
).round(4)

**Tension with the Sharpe story.** The LLM block contributes +0.0177
Δ AUC in 2024 but **−0.0021** in 2025 — yet the 2025 ΔSharpe (+1.07)
is an order of magnitude larger than 2024 (+0.13). The two facts
reconcile only if the 2025 lift is driven by a small number of
extreme-probability events whose realisations resolve favourably,
not by improved overall ranking. This is a tail-bucket effect, and
it is more fragile than aggregate AUC. The "underpowered finding"
framing in §4 of the paper is consistent with this picture: a
strategy whose edge concentrates in a handful of events per year
will fail standard significance tests even when the per-event
prediction is informative on average.

## 5. Outputs for the paper appendix

Save per-feature and block-level tables as CSV. These are what feed
Appendix D.3 of the paper.

In [ ]:
out_dir = config.PROJECT_ROOT / 'outputs' / 'importance'
out_dir.mkdir(parents=True, exist_ok=True)

per_feature = build_per_feature_table(perm_pool, feats, pipe=pipe)
per_feature.to_csv(out_dir / 'perm_importance_pooled.csv', index=False)
blk.to_csv(out_dir / 'block_summary.csv', index=False)
for year, df in per_year.items():
    df.to_csv(out_dir / f'perm_importance_{year}.csv', index=False)

print('Saved to:', out_dir)
print('Files:')
for fp in sorted(out_dir.iterdir()):
    print(f'  {fp.name}')